# Movie Night: Build a Simple Recommender
**Data Science Studio II | 45–60 minutes | Individual or pairs**

How could a streaming service answer “What should I watch if I liked this?” Build a **content-based recommender** using movie genres and cosine similarity.

By the end, you will encode item features, rank candidates, filter a recommendation list, and explain what your system can and cannot infer. You should already be comfortable with pandas and Python functions.

Complete each **TODO** and the written responses. Run cells in order. Submit this notebook with your code, visible outputs, and explanations. Starter cells deliberately raise `NotImplementedError` until completed.

## 1. Download and inspect the data (8 minutes)
1. Visit [GroupLens MovieLens Latest Datasets](https://grouplens.org/datasets/movielens/latest/).
2. Download **Small**: [ml-latest-small.zip](https://files.grouplens.org/datasets/movielens/ml-latest-small.zip).
3. Unzip it. Place the `ml-latest-small` folder beside this notebook. 
4. Read the dataset README and usage terms. Dataset reference: F. Maxwell Harper and Joseph A. Konstan (2015), *The MovieLens Datasets: History and Context*, https://doi.org/10.1145/2827872.

We only need `movies.csv`: `movieId`, `title`, and `genres`. Other files contain ratings and tags, but this activity does not use them. Genres describe the **items**; this is not a user–item utility matrix.

In [1]:
# Run once if packages are missing, then rerun the imports:
# %pip install pandas numpy scikit-learn
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

In [3]:
# TODO: Load movies.csv into a dataframe, and display its first five rows.
movies = pd.read_csv('/Users/madelinefrank/Downloads/Studio 2/ml-latest-small/movies.csv')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


**Pause and think:** Inspect the genres column. How are multiple genres stored? Why should `movieId` be an identifier rather than a numeric similarity feature?

**Your response:** Mutiple genres are stored in the same column with | seperating them. 'movieId' identifies which movie we are talking about. The number is an identifier. 

## 2. Represent movies as feature vectors (10 minutes)
The lecture represented anime using columns such as Action, Adventure, and Drama. Do the same for movies: a genre gets **1** if it is present, and **0** otherwise. This is **multi-hot encoding**, since a movie can have several genres.

For example, with columns `[Adventure, Comedy, Drama]`, an adventure/comedy movie becomes `[1, 1, 0]`.

Treat `(no genres listed)` as missing metadata, not a real genre. For this small activity, exclude those movies. Reset the row index so DataFrame rows and feature-array positions stay aligned.

In [6]:
# Provided: keep usable genre metadata and align row positions.
movies = movies.dropna(subset=["movieId", "title", "genres"]).copy()
movies = movies.loc[movies["genres"] != "(no genres listed)"].reset_index(drop=True)

# TODO: Create a binary DataFrame named genre_features.
# Hint: movies["genres"].str.get_dummies(sep="|")
# Display its first five rows and its shape.


In [7]:
genre_features = movies["genres"].str.get_dummies(sep="|")
display(genre_features.head())

,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


**Your response:** Choose one movie. Explain what its row of 0s and 1s represents. How does this matrix differ from the lecture's utility matrix?

For the first movie a 0 in Action, Crime, Documentary, Drama, Film- Noir, Horror, IMAX, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western means that the movie is not a part of those genres. The 1 in Adventure, Animation, Children, Comedy, Fantasy means that movie is apart of those genres. This matrix has more genre categories than the one in the lecture. 

## 3. Measure similarity (8 minutes)
Cosine similarity compares vector directions:

$$\operatorname{sim}(x,y)=\frac{x\cdot y}{\|x\|\|y\|}.$$

For these nonnegative genre vectors, the result ranges from 0 to 1. A score of 0 means no shared genres; 1 means identical genre sets in this binary representation. It is **not a predicted star rating or a probability of enjoyment**.

Before running the next cell, predict the similarity of `[1, 1, 0]` and `[1, 0, 1]`. Explain your prediction.

In [ ]:
# TODO: Use cosine_similarity to compare these two toy movies.
# Hint: each input needs to be 2D, e.g., [movie_a].
movie_a = [1, 1, 0]
movie_b = [1, 0, 1]


0.4999999999999999


In [8]:
movie_a = np.array([1,1,0])
movie_b = np.array([1,0,1])

similarity = cosine_similarity([movie_a], [movie_b])
print(f"Cosine similarity between movie_a and movie_b: {similarity[0][0]:.4f}")

Cosine similarity between movie_a and movie_b: 0.5000


## 4. Recommend five movies (15 minutes)
Use the provided helper to find a movie you like. Copy its `movieId`, which avoids ambiguity from similar titles.

Our pipeline follows the lecture: **choose a liked item → generate candidates → score and rank → filter → top-N items**. The selected movie is a simple explicit preference signal. We do not need other users' ratings.

In `recommend_movies`, compare **one selected row against all rows**; there is no need to build a large all-pairs similarity matrix. Exclude the selected movie and candidates with similarity 0. Break tied scores by `movieId` so results are reproducible; this tie-break does not imply higher quality.

In [9]:
# Provided: search titles without treating punctuation as a regular expression.
def find_movies(text):
    return movies.loc[
        movies["title"].str.contains(text, case=False, regex=False),
        ["movieId", "title", "genres"]
    ]

display(find_movies("Toy Story"))
# TODO: Search for a title of your choice.

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
7355,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX


### Turning similarities into recommendations

Our function takes a movie someone likes and finds other movies with similar genres.

- **`movie_id`** identifies the movie the user likes.
- **`n`** is the maximum number of recommendations to return.

The function will follow these steps:

1. **Find the selected movie's genre vector.**  
   Retrieve its row from `genre_features`. This row describes the movie using
   0s and 1s for each genre.

2. **Compare that vector with every movie.**  
   Use `cosine_similarity` to produce one similarity score per movie.
   Higher scores mean more similar genre combinations.

3. **Attach the scores to the movie information.**  
   Create a copy of `movies` and add a `similarity` column so we can see
   each title alongside its score.

4. **Remove unsuitable candidates.**  
   Exclude the selected movie—we do not want to recommend it to itself!
   Also remove movies with a score of 0, which share no genres with it.

5. **Rank and return the recommendations.**  
   Sort from highest to lowest similarity and return the first `n` movies.
   When scores tie, sort by movie ID to keep the order consistent.
   A lower movie ID does not mean a better recommendation.

In [10]:
def recommend_movies(movie_id, n=5):
    """Return up to n other movies ranked by genre cosine similarity."""

    # Provided: find the row containing the selected movie.
    # movieId is an identifier, which is not necessarily its row position.
    matches = movies.index[movies["movieId"] == movie_id].tolist()

    if not matches:
        raise ValueError("Movie ID not found. Use find_movies() to choose an ID.")
    if not isinstance(n, int) or n < 1:
        raise ValueError("n must be a positive integer.")

    selected_index = matches[0]

    # STEP 1: Retrieve the selected movie's genre features.
    # Double brackets keep the result as a table with one row.
    # cosine_similarity expects a 2D table, even for one movie.
    selected_features = genre_features.iloc[[selected_index]]

    # TODO 1: Compare selected_features against all rows in genre_features.
    # Use cosine_similarity(table_1, table_2).
    # The result will be a table with 1 row and one column per movie.
    similarities = cosine_similarity(selected_features, genre_features)

    # Provided: convert that single row into a 1D array.
    # Example: [[1.0, 0.4, 0.0]] becomes [1.0, 0.4, 0.0].
    scores = similarities.ravel()

    # Provided: copy the movie information so we do not change movies.
    candidates = movies.copy()

    # TODO 2: Add scores as a new column named "similarity".
    # The scores follow the same row order as movies.
    candidates["similarity"] = scores

    # TODO 3: Keep movies whose movieId differs from movie_id.
    # Hint: candidates.loc[candidates["movieId"] != movie_id]
    candidates = candidates.loc[candidates["movieId"] != movie_id]

    # TODO 4: Keep only movies with similarity greater than 0.
    candidates = candidates[candidates["similarity"] > 0]

    # Provided: rank by highest similarity first.
    # For tied scores, use movieId in ascending order.
    ranked = candidates.sort_values(
        ["similarity", "movieId"],
        ascending=[False, True]
    )

    # TODO 5: Return the first n rows.
    # Hint: use .head(n).
    return ranked.head(n)

In [12]:
# TODO: Replace 1 with the ID you chose; inspect the recommendations.
seed_id = 27
results = recommend_movies(seed_id, n=5)
display(movies.loc[movies["movieId"] == seed_id])
display(results)

,movieId,title,genres
26,27,Now and Then (1995),Children|Drama


,movieId,title,genres,similarity
32,34,Babe (1995),Children|Drama,1.0
72,80,"White Balloon, The (Badkonake sefid) (1995)",Children|Drama,1.0
207,241,Fluke (1995),Children|Drama,1.0
226,262,"Little Princess, A (1995)",Children|Drama,1.0
464,531,"Secret Garden, The (1993)",Children|Drama,1.0


### Explain one recommendation
Choose one returned movie. List the genres it shares with your seed movie, and explain how the shared features influenced the score. If several movies have score 1, what can the system distinguish between them?

**Your response:** The movie I chose is "Now and Then". They share the genres Children and Drama. Since they shared these genres and is only these 2 genres, it influenced the score to be similar. If several movies have the score 1 in similarity, they can use the year to distinguish between them. 

## 5. Check behavior and investigate limitations (10 minutes)
The checks below verify basic program behavior. Passing them does **not** establish that the recommendations are useful.

In [13]:
# Provided: run after completing your function.
check_id = int(movies.iloc[0]["movieId"])
check = recommend_movies(check_id, n=5)
assert len(check) <= 5
assert check_id not in check["movieId"].values
assert check["movieId"].is_unique
assert check["similarity"].between(0, 1 + 1e-12).all()
assert (check["similarity"] > 0).all()
assert check["similarity"].is_monotonic_decreasing
assert np.isclose(cosine_similarity([[1, 1, 0]], [[1, 0, 1]])[0, 0], 0.5)
print("Basic behavior checks passed. Now evaluate the results yourself.")

Basic behavior checks passed. Now evaluate the results yourself.


Run your recommender for **two seeds with different genre profiles**. Display both lists, then answer:

1. Do the recommendations seem useful? Explain one reasonable match and one limitation using actual outputs. If you do not know the movies, distinguish genre plausibility from evidence of enjoyment.
2. Where do you see **overspecialization**? Name one additional feature that could improve the recommendations.
3. Can a new movie with genres be recommended without ratings? What happens for a new user who has not selected a liked movie? What about a movie with no genre metadata?
4. Does this system predict ratings? Would RMSE make sense for its current output? Explain. Describe how you could instead collect feedback about top-5 usefulness. Our inspection is a qualitative sanity check, not a formal offline evaluation.

**Your responses:** TODO

In [ ]:
# TODO: Choose two IDs with different genres and display their recommendations.
# display(recommend_movies(...))
# display(recommend_movies(...))

## Optional extension: A profile from several liked movies
Choose three movie IDs. Average their genre vectors to form a **user profile**, compare that profile against every movie, and exclude **all** three liked movies. Here a profile value of 2/3 means that genre appears in two of the three selected movies.

Compare this list with your single-seed recommendations. How did the representation change the results?

## Submission checklist
- Completed TODO code and visible outputs.
- A working top-5 recommender and the behavior checks.
- Two seeds with different genre profiles and their recommendation lists.
- Written explanations and reflections.

**Scope:** We use genres only. This is content-based filtering, not collaborative filtering. Context-aware recommendation would additionally consider circumstances such as time, location, or mood.